# ontodocker/fuseki API calls via `OntodockerClient`

### read in address and bearer token

Token and address should be stored externally, not directly writte to published notebooks. Howits done is not relevant for `courier`; they have to be provided as `str`.

In [1]:
import json
with open("secrets/registry.json") as json_file:
    registry = json.load(json_file)

ontodocker_token = registry["mpi_susmat"]["services"]["ontodocker"]["token"]
ontodocker_address = registry["mpi_susmat"]["services"]["ontodocker"]["address"]

### instantiate an `OntodockerClient`

In [2]:
from courier.services.ontodocker import OntodockerClient
ontodocker = OntodockerClient(address=ontodocker_address, token=ontodocker_token)

## fundamental actions

### get a list of all datasets on an instance

In [3]:
datasets = ontodocker.datasets.list()
print(datasets)

['pmdco2_tto_example']


### create a dataset

In [4]:
test_name="test_dataset"
ontodocker.datasets.create(name=test_name)

'"Dataset name test_dataset created"'

### populate a dataset

#### 1) turtlefile from disk

In [5]:
test_ttlfile="./test_data/test_dataset.ttl" 
ontodocker.datasets.upload_turtlefile(name=test_name, turtlefile=test_ttlfile)

'"Upload succeeded { \\n  \\"count\\" : 13 ,\\n  \\"tripleCount\\" : 13 ,\\n  \\"quadCount\\" : 0\\n}\\n"'

#### 2) `rdflib.Graph` instance

In [6]:
from rdflib import Graph

In [7]:
test_graph = Graph()
test_graph.parse("./test_data/test_dataset.ttl", format="turtle")

<Graph identifier=Nc055c6fe6630439c8aa98e22f24721a2 (<class 'rdflib.graph.Graph'>)>

In [8]:
ontodocker.datasets.upload_graph(name=test_name, graph=test_graph)

'"Upload succeeded { \\n  \\"count\\" : 13 ,\\n  \\"tripleCount\\" : 13 ,\\n  \\"quadCount\\" : 0\\n}\\n"'

### SPARQL-query a dataset

In [9]:
test_query = f"""
    SELECT ?s ?p ?o
    WHERE {{ ?s ?p ?o }}
    """

#### send query to the endpoint of a dataset
... and return the raw JSONish response (per default `Accept` header)

In [10]:
result = ontodocker.sparql.query_raw(dataset=test_name, query=test_query)
print(type(result))
result

<class 'str'>


'{"head":{"vars":["s","p","o"]},"results":{"bindings":[{"s":{"type":"uri","value":"http://example.org/Person"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#Class"}},{"s":{"type":"uri","value":"http://example.org/Person"},"p":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#label"},"o":{"type":"literal","xml:lang":"en","value":"Person"}},{"s":{"type":"uri","value":"http://example.org/Project"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#Class"}},{"s":{"type":"uri","value":"http://example.org/Project"},"p":{"type":"uri","value":"http://www.w3.org/2000/01/rdf-schema#label"},"o":{"type":"literal","xml:lang":"en","value":"Project"}},{"s":{"type":"uri","value":"http://example.org/worksOn"},"p":{"type":"uri","value":"http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},"o":{"type":"uri","value":

#### send query to the endpoint of a dataset and return the response mapped to a `pandas.DataFrame`
... makes sense for typical `SELECT` queries 

In [11]:
result = ontodocker.sparql.query_df(dataset=test_name, query=test_query, columns= ["subject", "predicate", "object"])
print(type(result))
result

<class 'pandas.DataFrame'>


,subject,predicate,object
0,http://example.org/Person,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
1,http://example.org/Person,http://www.w3.org/2000/01/rdf-schema#label,Person
2,http://example.org/Project,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Class
3,http://example.org/Project,http://www.w3.org/2000/01/rdf-schema#label,Project
4,http://example.org/worksOn,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2000/01/rdf-schema#Property
5,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#label,works on
6,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#domain,http://example.org/Person
7,http://example.org/worksOn,http://www.w3.org/2000/01/rdf-schema#range,http://example.org/Project
8,http://example.org/alice,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/Person
9,http://example.org/alice,http://www.w3.org/2000/01/rdf-schema#label,Alice


#### optional: get the SPARQL endpoint of a dataset

In [12]:
test_endpoint = ontodocker.sparql.endpoint(dataset=test_name)
print(test_endpoint)

https://ontodocker.pmds-mpcdf-2.pyiron.com/api/v1/jena/test_dataset/sparql


### download the turtlefile of a dataset

In [13]:
_ = ontodocker.datasets.download_turtle(name=test_name, filename="test_turtle_dump.ttl") 

Wrote test_turtle_dump.ttl.


# Delete dataset

In [14]:
ontodocker.datasets.delete(name="test_dataset")

'"Dataset name test_dataset destroyed"'